# Routes from Jeddah Islamic Port to key Saudi destinations
1. Download the Geofabrik GCC extract and filter it to major roads (low memory).
2. Build a routable graph with OSMnx, with a car and a heavy-truck speed profile.
3. Write the GIS layers (one GeoPackage for QGIS), including empty layers for the shipment, gate and constraint data you are still collecting.
4. Route from the port to every destination (cities now; your warehouses/factories as you collect them).
5. Export route lines + table for QGIS and the paper.

CPU runtime is fine. The download is a few hundred MB, so it runs once per Colab session.

In [ ]:
!pip -q install osmnx
!apt-get -qq update && apt-get -qq install -y osmium-tool > /dev/null

## 0. Settings

In [ ]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = Path("/content/drive/MyDrive/jeddah_port")
except ImportError:  # running outside Colab
    OUT_DIR = Path("out")
OUT_DIR.mkdir(parents=True, exist_ok=True)

GPKG = OUT_DIR / "jeddah_port_gis.gpkg"  # open this in QGIS
SITES_CSV = OUT_DIR / "sites.csv"        # optional: your warehouses/factories, same columns as `dest` below
CRS_WORK = "EPSG:2318"                   # Ain el Abd / Aramco Lambert: metres, covers all of KSA
TRUCK_MAX_KPH = 80                       # heavy-vehicle cap: replace with the figure from the regulation you cite
SCENARIO = "baseline"                    # label for this run; routes from other scenarios are kept for comparison

## 1. National road extract (major roads only)

In [ ]:
%%bash
set -e
if [ -f saudi_major.osm ]; then echo "already built"; ls -lh saudi_major.osm; exit 0; fi

# Geofabrik covers Saudi Arabia inside the 'GCC States' extract (Asia). If this URL 404s,
# open download.geofabrik.de/asia.html and copy the current GCC States .osm.pbf link.
URL="https://download.geofabrik.de/asia/gcc-states-latest.osm.pbf"
wget -q -O gcc.osm.pbf "$URL"
# Snapshot date, to cite with the results
osmium fileinfo -g header.option.osmosis_replication_timestamp gcc.osm.pbf > osm_date.txt

# Keep only major roads, then trim to a Saudi bounding box (min lon, min lat, max lon, max lat).
# The box also clips Kuwait, Bahrain, Qatar and western UAE; harmless for these routes.
osmium tags-filter gcc.osm.pbf w/highway=motorway,motorway_link,trunk,trunk_link,primary,primary_link,secondary,secondary_link -o major.osm.pbf --overwrite
osmium extract -b 34.4,16.0,55.8,32.4 major.osm.pbf -o saudi_major.osm.pbf --overwrite
osmium cat saudi_major.osm.pbf -o saudi_major.osm --overwrite
ls -lh saudi_major.osm; cat osm_date.txt

## 2. Build the graph
Two speed profiles per road segment:
* `travel_time`: car free-flow, from `maxspeed` or the road-class average.
* `truck_tt`: the same speed capped at `TRUCK_MAX_KPH`.

The graph is then projected to metres so snapping distances are real.

In [ ]:
import osmnx as ox
import geopandas as gpd
import pandas as pd
import networkx as nx
import pyogrio
from shapely.geometry import Point, LineString
from shapely.ops import substring, linemerge

G = ox.graph_from_xml("saudi_major.osm", simplify=True, retain_all=True)
G = ox.truncate.largest_component(G, strongly=True)  # drop disconnected fragments
G = ox.add_edge_speeds(G)                            # km/h; fills missing maxspeed from road class
G = ox.add_edge_travel_times(G)                      # seconds, car
for u, v, k, d in G.edges(keys=True, data=True):
    d["truck_tt"] = d["length"] / (min(d["speed_kph"], TRUCK_MAX_KPH) / 3.6)
G = ox.project_graph(G, to_crs=CRS_WORK)

OSM_DATE = Path("osm_date.txt").read_text().strip() if Path("osm_date.txt").exists() else "unknown"
print(f"{len(G.nodes):,} nodes, {len(G.edges):,} edges, OSM snapshot {OSM_DATE}")

## 3. Port and destinations
Every point has an ID; the IDs are what link shipments, routes and destinations in QGIS.
City coordinates are approximate centres: verify them (OSM / Google Maps) and set `verified = True`.

Add warehouses and factories in `sites.csv` (in `OUT_DIR`) with the same columns, `type` = `warehouse` or `factory`, and a source for each coordinate. They are picked up on the next run.

In [ ]:
port = pd.DataFrame([
    # port_id, name, lat, lon, source, verified
    ("JIP", "Jeddah Islamic Port", 21.485, 39.165, "approx; verify vs NGA World Port Index / Mawani", False),
], columns=["port_id", "name", "lat", "lon", "source", "verified"])

dest = pd.DataFrame([
    # dest_id, name, type, lat, lon, source, verified
    ("MAKKAH",   "Makkah",   "city", 21.42, 39.83, "approx city centre", False),
    ("MADINAH",  "Madinah",  "city", 24.47, 39.61, "approx city centre", False),
    ("TAIF",     "Taif",     "city", 21.27, 40.42, "approx city centre", False),
    ("RABIGH",   "Rabigh",   "city", 22.80, 39.03, "approx city centre", False),
    ("YANBU",    "Yanbu",    "city", 24.09, 38.06, "approx city centre", False),
    ("RIYADH",   "Riyadh",   "city", 24.71, 46.68, "approx city centre", False),
    ("BURAYDAH", "Buraydah", "city", 26.33, 43.97, "approx city centre", False),
    ("DAMMAM",   "Dammam",   "city", 26.43, 50.10, "approx city centre", False),
    ("JUBAIL",   "Jubail",   "city", 27.00, 49.66, "approx city centre", False),
    ("ABHA",     "Abha",     "city", 18.22, 42.50, "approx city centre", False),
    ("JAZAN",    "Jazan",    "city", 16.89, 42.55, "approx city centre", False),
    ("TABUK",    "Tabuk",    "city", 28.38, 36.57, "approx city centre", False),
], columns=["dest_id", "name", "type", "lat", "lon", "source", "verified"])

if SITES_CSV.exists():
    dest = pd.concat([dest, pd.read_csv(SITES_CSV)], ignore_index=True)
assert dest.dest_id.is_unique, "dest_id values must be unique"
dest

## 4. GIS layers
All layers go into one GeoPackage (`GPKG`), in EPSG:4326.

| layer | geometry | owner | content |
|---|---|---|---|
| `port` | point | notebook | origin port(s) |
| `destinations` | point | notebook | cities + `sites.csv`; after a `baseline` run also its distances and hours |
| `routes` | line | notebook | one row per scenario, destination and profile (`car` / `truck`), written in step 6 |
| `roads` | line | notebook | the routable network: one row per directed segment with the car / truck seconds the router used |
| `port_gates` | point | you | truck gates / terminals |
| `constraints` | point | you | truck bans, checkpoints, weigh stations, roadworks |
| `shipments` | table (no geometry) | you | one row per shipment; join to `routes` / `destinations` on `dest_id` |

Notebook-owned layers are rebuilt on every run, except that `routes` only replaces rows of the current `SCENARIO`. Your layers are created empty once and **never overwritten**, so you can fill them in QGIS without losing edits.

In [ ]:
def to_points(df):
    return gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.lon, df.lat), crs="EPSG:4326")

to_points(port).to_file(GPKG, layer="port")
to_points(dest).to_file(GPKG, layer="destinations")

TEMPLATES = {  # layer: (geometry type or None, {column: dtype})
    "port_gates": ("Point", {
        "gate_id": "str", "port_id": "str", "name": "str", "gate_type": "str",
        "source": "str", "verified": "bool"}),
    "constraints": ("Point", {
        "constraint_id": "str", "kind": "str", "name": "str", "applies_to": "str",
        "time_window": "str", "description": "str", "source": "str", "verified": "bool"}),
    "shipments": (None, {
        "shipment_id": "str", "date": "datetime64[ms]", "port_id": "str", "dest_id": "str",
        "commodity": "str", "hs_code": "str", "tonnes": "float", "teu": "float",
        "port_dwell_days": "float", "customs_days": "float", "inland_cost_sar": "float",
        "source": "str"}),
}
existing = set(gpd.list_layers(GPKG)["name"])
for name, (geom_type, cols) in TEMPLATES.items():
    if name in existing:
        continue
    empty = gpd.GeoDataFrame({c: pd.Series(dtype=t) for c, t in cols.items()},
                             geometry=gpd.GeoSeries([], crs="EPSG:4326"))
    if geom_type is None:
        empty = pd.DataFrame(empty.drop(columns="geometry"))
    pyogrio.write_dataframe(empty, GPKG, layer=name, geometry_type=geom_type)
    print("created empty layer", name)
gpd.list_layers(GPKG)

## 5. Route from the port to every destination
Points are snapped to the nearest road **segment** (not the nearest intersection, which can be tens of km away on desert highways), and routes start and end at that spot on the segment. `origin_snap_m` / `dest_snap_m` give the straight-line distance from the point to the road; a large value means the site's access road is not in the major-road network, so treat its distance as approximate.

In [ ]:
def edge_line(G, u, v, k):
    d = G.edges[u, v, k]
    if "geometry" in d:
        return d["geometry"]
    return LineString([(G.nodes[u]["x"], G.nodes[u]["y"]), (G.nodes[v]["x"], G.nodes[v]["y"])])


def snap_points(G, xs, ys):
    """Snap points onto their nearest road segment. For each point returns
    ([(u, v, k, fraction along edge), ...], snap distance in m); the list holds
    both directions of a two-way road."""
    edges, dists = ox.distance.nearest_edges(G, xs, ys, return_dist=True)
    out = []
    for (u, v, _), dist, x, y in zip(edges, dists, xs, ys):
        p, on = Point(x, y), []
        for a, b in {(u, v), (v, u)}:
            for k in G[a][b] if G.has_edge(a, b) else []:
                line = edge_line(G, a, b, k)
                if line.distance(p) <= dist + 1:  # same road, not a parallel one
                    on.append((a, b, k, line.project(p, normalized=True)))
        out.append((on, float(dist)))
    return out


def route_from(G, origin, targets, weight):
    """Fastest routes by `weight` from one snapped origin to many snapped targets
    ({id: snap}). Returns {id: dict(geometry, distance_m, seconds)}."""
    # Leave the origin segment through its far end, paying for the part still ahead.
    start = {}
    for a, b, k, f in origin[0]:
        c = (1 - f) * G.edges[a, b, k][weight]
        if b not in start or c < start[b][0]:
            start[b] = (c, (a, b, k, f))
    SRC = "__origin__"
    G.add_node(SRC)
    for b, (c, _) in start.items():
        G.add_edge(SRC, b, **{weight: c})
    try:
        cost, paths = nx.single_source_dijkstra(G, SRC, weight=weight)
    finally:
        G.remove_node(SRC)

    out = {}
    for tid, (on, _) in targets.items():
        reach = [(cost[a] + g * G.edges[a, b, k][weight], a, b, k, g) for a, b, k, g in on if a in cost]
        if not reach:
            out[tid] = dict(geometry=None, distance_m=None, seconds=None)
            continue
        secs, a, b, k, g = min(reach)
        nodes = paths[a][1:]
        oa, ob, ok_, f = start[nodes[0]][1]
        parts = [substring(edge_line(G, oa, ob, ok_), f, 1, normalized=True)]
        dist = (1 - f) * G.edges[oa, ob, ok_]["length"]
        for u, v in zip(nodes[:-1], nodes[1:]):
            kk = min(G[u][v], key=lambda key: G[u][v][key][weight])
            parts.append(edge_line(G, u, v, kk))
            dist += G.edges[u, v, kk]["length"]
        parts.append(substring(edge_line(G, a, b, k), 0, g, normalized=True))
        dist += g * G.edges[a, b, k]["length"]
        parts = [p for p in parts if p.geom_type == "LineString" and p.length > 0]
        out[tid] = dict(geometry=linemerge(parts), distance_m=dist, seconds=secs)
    return out


PROFILES = {"car": "travel_time", "truck": "truck_tt"}

o = to_points(port.iloc[[0]]).to_crs(CRS_WORK).geometry.iloc[0]
d = to_points(dest).to_crs(CRS_WORK).geometry
snaps = snap_points(G, [o.x, *d.x], [o.y, *d.y])
origin, targets = snaps[0], dict(zip(dest.dest_id, snaps[1:]))

recs = []
for profile, weight in PROFILES.items():
    for did, r in route_from(G, origin, targets, weight).items():
        ok = r["geometry"] is not None
        recs.append(dict(
            route_id=f"{SCENARIO}-{port.port_id[0]}-{did}-{profile}", scenario=SCENARIO,
            port_id=port.port_id[0], dest_id=did, profile=profile,
            distance_km=round(r["distance_m"] / 1000, 1) if ok else None,
            hours=round(r["seconds"] / 3600, 2) if ok else None,
            origin_snap_m=round(origin[1]), dest_snap_m=round(targets[did][1]),
            truck_max_kph=TRUCK_MAX_KPH if profile == "truck" else None,
            osm_date=OSM_DATE, geometry=r["geometry"]))
routes = gpd.GeoDataFrame(recs, geometry="geometry", crs=CRS_WORK).to_crs("EPSG:4326")

r = pd.DataFrame(routes.drop(columns="geometry"))
pick = lambda p: r[r.profile == p][["dest_id", "distance_km", "hours"]].add_prefix(p + "_").rename(columns={p + "_dest_id": "dest_id"})
result = (dest[["dest_id", "name", "type"]]
          .merge(pick("truck"), on="dest_id").merge(pick("car"), on="dest_id")
          .merge(r[r.profile == "truck"][["dest_id", "dest_snap_m"]], on="dest_id"))
result["truck_extra_hours"] = (result.truck_hours - result.car_hours).round(2)
print(f"Port snapped {origin[1]:.0f} m to the network")
result

Times are free-flow estimates from speed limits and road class. `truck_extra_hours` is the cost of the truck speed cap alone; congestion, city truck-entry bans, port dwell, customs and checkpoints are not in it yet (record them in `constraints` and `shipments`).

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 9))
ox.graph_to_gdfs(G, nodes=False).plot(ax=ax, color="#cccccc", linewidth=0.4)
routes[routes.profile == "truck"].dropna(subset=["geometry"]).to_crs(CRS_WORK).plot(ax=ax, color="#c0392b", linewidth=1.6)
to_points(port).to_crs(CRS_WORK).plot(ax=ax, color="black", markersize=60, zorder=5)
dp = to_points(dest).to_crs(CRS_WORK)
dp.plot(ax=ax, color="#2c3e50", markersize=18, zorder=5)
for name, p in zip(dp.name, dp.geometry):
    ax.annotate(name, (p.x, p.y), fontsize=8, xytext=(3, 3), textcoords="offset points")
ax.set_title("Fastest truck routes from Jeddah Islamic Port")
ax.set_axis_off()
plt.show()

## 6. Export

In [ ]:
# Routes: replace this scenario's rows and keep the others, so scenarios can be compared side by side
if "routes" in set(gpd.list_layers(GPKG)["name"]):
    old = gpd.read_file(GPKG, layer="routes")
    old = old[old["scenario"] != SCENARIO] if "scenario" in old else old.iloc[0:0]
    routes_all = pd.concat([old, routes], ignore_index=True)
else:
    routes_all = routes
routes_all.to_file(GPKG, layer="routes", promote_to_multi=True)

if SCENARIO == "baseline":  # city points carry the baseline results, for labels and clicking in QGIS
    to_points(dest.merge(result.drop(columns=["name", "type"]), on="dest_id", how="left")).to_file(GPKG, layer="destinations")
result.assign(scenario=SCENARIO).to_csv(OUT_DIR / f"route_table_{SCENARIO}.csv", index=False)

# The routable network itself: one row per directed segment (u -> v) with the costs the router used
edges = ox.graph_to_gdfs(G, nodes=False).reset_index()
first = lambda x: x[0] if isinstance(x, list) else x
col = lambda c: edges[c].map(first) if c in edges else None
roads = gpd.GeoDataFrame(dict(
    u=edges.u, v=edges.v, key=edges.key,
    highway=col("highway"), road_class=col("highway").str.replace("_link", ""),
    name=col("name"), ref=col("ref"), oneway=edges.oneway,
    maxspeed_tagged=edges["maxspeed"].notna() if "maxspeed" in edges else False,
    speed_kph=edges.speed_kph.round(1), length_m=edges.length.round(1),
    car_s=edges.travel_time.round(1), truck_s=edges.truck_tt.round(1), osm_date=OSM_DATE,
    geometry=edges.geometry.simplify(10)),  # 10 m simplification is for display; costs use the full geometry
    crs=CRS_WORK).to_crs("EPSG:4326")
roads.to_file(GPKG, layer="roads")
print("Saved to", GPKG)
gpd.list_layers(GPKG)

## 7. Seaports and chokepoints (IMF PortWatch)
Daily ship calls and estimated trade (tonnes) for every Saudi seaport, and daily transits through the chokepoints (Hormuz, Bab el-Mandeb, Suez, ...), from AIS ship tracking. PortWatch updates weekly; each run downloads the latest and keeps a copy in `OUT_DIR/portwatch`, which is used if the download fails.

Writes the `seaports` layer: each port's monthly trade in the 12 months before `SHOCK_DATE` and the change since. Vessel type stands in for commodity (tanker: crude/refined/liquid chemicals; dry bulk: fertilizer, minerals, metals; container: manufactured goods).

In [ ]:
import requests

PW = "https://services9.arcgis.com/weJ1QsnbMYJlCHdG/arcgis/rest/services"
PW_DIR = OUT_DIR / "portwatch"
PW_DIR.mkdir(exist_ok=True)
SHOCK_DATE = pd.Timestamp("2026-03-01")  # Hormuz transits collapse; compare each port before / since


def bulk_csv(layer_url, dest):
    """Download a layer's full CSV export via its ArcGIS Hub item ID. Returns True on success."""
    item = requests.get(layer_url, params={"f": "json"}, timeout=60).json().get("serviceItemId")
    if not item:
        return False
    for url in (f"https://portwatch.imf.org/api/download/v1/items/{item}/csv?layers=0",
                f"https://hub.arcgis.com/api/download/v1/items/{item}/csv?redirect=true&layers=0"):
        with requests.get(url, stream=True, timeout=600) as r:
            head = r.raw.read(200, decode_content=True)
            if r.status_code != 200 or head.lstrip().startswith(b"<") or b"," not in head:
                continue
            with open(dest, "wb") as f:
                f.write(head)
                for block in r.iter_content(1 << 20):
                    f.write(block)
        return True
    return False


def refresh(services, dest, keep=lambda c: c):
    """Latest copy of a PortWatch dataset (first service name that answers), else the saved one."""
    tmp = Path("pw_download.csv")
    for service in services:
        try:
            if bulk_csv(f"{PW}/{service}/FeatureServer/0", tmp):
                pd.concat(keep(c) for c in pd.read_csv(tmp, chunksize=200_000)).to_csv(dest, index=False)
                tmp.unlink()
                print("downloaded", dest.name)
                break
        except requests.RequestException as e:
            print(f"{service}: {e.__class__.__name__}")
    else:
        if not dest.exists():
            raise RuntimeError(f"could not download {services} and no saved copy at {dest}")
        print("using saved copy of", dest.name)
    return pd.read_csv(dest)


ports_list = refresh(("PortWatch_ports_database", "Ports_database", "PortWatch_Ports", "Ports"),
                     PW_DIR / "portwatch_ports_list.csv")
ports_daily = refresh(("Daily_Ports_Data",), PW_DIR / "portwatch_ports_daily_SAU.csv",
                      keep=lambda c: c[c.ISO3 == "SAU"])
choke = refresh(("Daily_Chokepoints_Data",), PW_DIR / "portwatch_chokepoints_daily.csv")

ports_daily["date"] = pd.to_datetime(ports_daily.date)
choke["date"] = pd.to_datetime(choke.date)
last_full_month = (ports_daily.date.max() + pd.Timedelta(days=1)).to_period("M").to_timestamp()  # exclusive
print(f"ports to {ports_daily.date.max():%Y-%m-%d} (complete months before {last_full_month:%Y-%m}), "
      f"chokepoints to {choke.date.max():%Y-%m-%d}")


def monthly(a, b):
    w = ports_daily[(ports_daily.date >= a) & (ports_daily.date < b)]
    months = max(len(pd.period_range(a, b - pd.Timedelta(days=1), freq="M")), 1)
    return w.groupby("portid")[["portcalls", "import", "export"]].sum() / months, w


before, w_before = monthly(SHOCK_DATE - pd.DateOffset(months=12), SHOCK_DATE)
since, _ = monthly(SHOCK_DATE, last_full_month)
exp_types = ["export_tanker", "export_dry_bulk", "export_container", "export_general_cargo", "export_roro"]
by_type = w_before.groupby("portid")[exp_types].sum()
main_export = by_type.idxmax(axis=1).str.replace("export_", "").where(by_type.sum(axis=1) > 0)

sea = ports_list[ports_list.ISO3 == "SAU"].set_index("portid")
stats = pd.DataFrame({
    "calls_per_month_before": before.portcalls.round(0), "calls_per_month_since": since.portcalls.round(0),
    "import_Mt_month_before": (before["import"] / 1e6).round(2), "import_Mt_month_since": (since["import"] / 1e6).round(2),
    "export_Mt_month_before": (before.export / 1e6).round(2), "export_Mt_month_since": (since.export / 1e6).round(2),
    "main_export_vessel": main_export})
for k in ("import", "export"):
    b, s = stats[f"{k}_Mt_month_before"], stats[f"{k}_Mt_month_since"]
    stats[f"{k}_change_pct"] = (100 * (s / b - 1)).where(b > 0.05).round(0)  # skip near-zero bases
sea = sea[["portname", "lat", "lon", "LOCODE", "share_country_maritime_import", "share_country_maritime_export",
           "industry_top1", "industry_top2", "industry_top3"]].join(stats)
sea["coast"] = (sea.lon < 45).map({True: "Red Sea", False: "Gulf"})
sea["shock_date"] = f"{SHOCK_DATE:%Y-%m-%d}"
sea["data_through"] = f"{ports_daily.date.max():%Y-%m-%d}"
seaports = to_points(sea.reset_index())
seaports.to_file(GPKG, layer="seaports")
seaports.drop(columns="geometry").sort_values("export_Mt_month_before", ascending=False)[
    ["portname", "coast", "main_export_vessel", "import_Mt_month_before", "import_change_pct",
     "export_Mt_month_before", "export_change_pct"]]

## 8. QGIS styles
Open `jeddah_port_map.qgz` (same folder as the GeoPackage) for the styled map: layer order, labels, basemap.
The GeoPackage also stores each layer's style, so dragging it into any QGIS project looks the same. Styles survive reruns; this cell only re-applies them from `styles/*.qml` if the GeoPackage was deleted and rebuilt.

In [ ]:
import sqlite3

con = sqlite3.connect(GPKG)
con.execute('''CREATE TABLE IF NOT EXISTS "layer_styles" ("id" INTEGER PRIMARY KEY AUTOINCREMENT NOT NULL,
    "f_table_catalog" TEXT(256), "f_table_schema" TEXT(256), "f_table_name" TEXT(256),
    "f_geometry_column" TEXT(256), "styleName" TEXT(30), "styleQML" TEXT, "styleSLD" TEXT,
    "useAsDefault" BOOLEAN, "description" TEXT, "owner" TEXT(30), "ui" TEXT(30),
    "update_time" DATETIME DEFAULT (strftime('%Y-%m-%dT%H:%M:%fZ','now')))''')
con.execute("INSERT OR IGNORE INTO gpkg_contents (table_name, data_type, identifier) "
            "VALUES ('layer_styles', 'attributes', 'layer_styles')")
styled = {r[0] for r in con.execute("SELECT f_table_name FROM layer_styles WHERE useAsDefault")}
layers = set(gpd.list_layers(GPKG)["name"])
for qml in sorted((OUT_DIR / "styles").glob("*.qml")):
    if qml.stem in layers and qml.stem not in styled:
        con.execute("INSERT INTO layer_styles (f_table_catalog, f_table_schema, f_table_name, f_geometry_column, "
                    "styleName, styleQML, styleSLD, useAsDefault, description, owner) "
                    "VALUES ('', '', ?, 'geom', 'jeddah_port', ?, '', 1, 'styled by notebook setup', '')",
                    (qml.stem, qml.read_text()))
        print("applied style:", qml.stem)
con.commit()
con.close()